# 环境与依赖

In [ ]:
import pandas as pd
pd.set_option("display.precision", 4)

import numpy as np
np.set_printoptions(precision=4, suppress=True)

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
from matplotlib import font_manager

import statsmodels.api as sm

In [ ]:
plt.rcParams["font.sans-serif"] = ["SimHei"] 
plt.rcParams["axes.unicode_minus"] = False     

# 第一步 得到干净的总股本数据

In [ ]:
# 导入csmar的总股本数据
total_share_csmar = pd.read_excel(r'\总股本与总市值.xlsx')
total_share_csmar['统计截止日期'] = pd.to_datetime(total_share_csmar['统计截止日期'], format='%Y-%m-%d')
total_share_csmar['证券代码'] = total_share_csmar['证券代码'].astype(str).str.zfill(6)
total_share_csmar.head()

In [ ]:
# 导入公告日期
announcement_date = pd.read_csv(r'\公告日期.csv')
announcement_date['公告日期'] = pd.to_datetime(announcement_date['公告日期'], format='%Y-%m-%d')
announcement_date['统计截止日期'] = pd.to_datetime(announcement_date['统计截止日期'], format='%Y-%m-%d')
announcement_date['证券代码'] = announcement_date['证券代码'].astype(str).str.zfill(6)
announcement_date = announcement_date[['证券代码','统计截止日期','公告日期']]
announcement_date.head()


In [ ]:
# 并入公告日期
total_share = pd.merge(total_share_csmar, announcement_date, on=['证券代码','统计截止日期'], how='left')
total_share.head()

In [ ]:
# 导入wind与csmar股票代码对照表
code_map = pd.read_excel(r'\wind与csmar股票代码对照表.xlsx')
code_map['证券代码'] = code_map['证券代码'].astype(str).str.zfill(6)
code_map.head()


In [ ]:
total_share = pd.merge(total_share, code_map, on='证券代码', how='left')
total_share.head(10)

In [ ]:


q = total_share.copy()
q['公告日期'] = pd.to_datetime(q['公告日期'], errors='coerce')
q['统计截止日期'] = pd.to_datetime(q['统计截止日期'], errors='coerce')

# 关键：右表键（公告日期）不能有空值
q = q.dropna(subset=['Code', '公告日期', '总股本']).sort_values(['Code', '公告日期'])

month_end = pd.DataFrame({'month_end': pd.date_range(q['公告日期'].min(), q['公告日期'].max(), freq='M')})

def to_monthly(g):
    g = g.dropna(subset=['公告日期'])  # 关键：组内再保险（防止仍有 NaT）
    if g.empty:
        return pd.DataFrame(columns=['month_end','Code','统计截止日期','总股本'])

    m = month_end.copy()
    m['Code'] = g['Code'].iloc[0]

    out = pd.merge_asof(
        m.sort_values('month_end'),
        g[['公告日期','统计截止日期','总股本']].sort_values('公告日期'),
        left_on='month_end', right_on='公告日期',
        direction='backward'
    )
    return out

monthly = (q.groupby('Code', group_keys=False)
             .apply(to_monthly)
             .dropna(subset=['总股本'])
             .reset_index(drop=True))


In [ ]:
total_share_monthly = monthly.sort_values(['Code', 'month_end'])
total_share_monthly.head()
total_share_monthly.to_csv(r'\total_share.csv', encoding='utf-8-sig', index=False)


# 第二步 并入筛选指标并对数据进行筛选，并计算数据覆盖度

In [ ]:
filter_df = pd.read_csv(r'\总市值与筛选指标.csv')
filter_df['总股本'] = pd.to_numeric(filter_df['总股本'], errors='coerce')
filter_df['月末收盘价'] = pd.to_numeric(filter_df['月末收盘价'], errors='coerce')
filter_df['涨停价'] = pd.to_numeric(filter_df['涨停价'], errors='coerce')
filter_df['跌停价'] = pd.to_numeric(filter_df['跌停价'], errors='coerce')
filter_df['总市值'] = filter_df['总股本'] * filter_df['月末收盘价']
filter_df.head()

In [ ]:
def clean_universe(df,
                   date_col='month_end',
                   risk_col='是否属于风险警示板',
                   navps_col='每股净资产',
                   ipo_days_col='上市天数',
                   close_col='月末收盘价',
                   up_col='涨停价',
                   down_col='跌停价',
                   min_ipo_days=120,
                   winsor_cols=None,
                   q=0.01,
                   drop_limit=True,
                   eps=1e-8):
    x = df.copy()

    # 1) 风险警示股
    if risk_col in x.columns:
        rc = x[risk_col]
        if rc.dtype == 'O':
            s = rc.astype(str).str.strip().str.upper()
            risk = s.isin(['1', 'TRUE', 'T', 'Y', 'YES', '是', 'ST', '*ST'])
        else:
            risk = rc.fillna(0).astype(float).ne(0)
        x = x.loc[~risk]

    # 2) 净资产为负（每股净资产<=0）
    if navps_col in x.columns:
        x = x.loc[x[navps_col] > 0]

    # 3) 次新股
    if ipo_days_col in x.columns:
        x = x.loc[x[ipo_days_col] >= min_ipo_days]

    # 4) 剔除涨跌停（调仓日不可交易）
    if drop_limit and all(c in x.columns for c in [close_col, up_col, down_col]):
        eps = 1e-8
        is_up = x[close_col].notna() & x[up_col].notna() & (x[close_col] >= x[up_col] - eps)
        is_dn = x[close_col].notna() & x[down_col].notna() & (x[close_col] <= x[down_col] + eps)
        x = x.loc[~(is_up | is_dn)]

    # 5) 横截面缩尾（左右各1%）
    if winsor_cols is None:
        num_cols = x.select_dtypes(include='number').columns.tolist()
        drop_like = ['价', '价格', '收盘', '涨停', '跌停']
        winsor_cols = [c for c in num_cols
                       if c != ipo_days_col and not any(k in c for k in drop_like)]

    if winsor_cols:
        def _clip(s):
            lo = s.quantile(q)
            hi = s.quantile(1 - q)
            return s.clip(lo, hi)
        x[winsor_cols] = x.groupby(date_col, group_keys=False)[winsor_cols].transform(_clip)

    return x


In [ ]:
cleaned_df = clean_universe(filter_df,
                   date_col='month_end',
                   risk_col='是否属于风险警示板',
                   navps_col='每股净资产',
                   ipo_days_col='上市天数',
                   close_col='月末收盘价',
                   up_col='涨停价',
                   down_col='跌停价',
                   min_ipo_days=252,
                   winsor_cols=['总市值', '总股本'],
                   q=0.01,
                   drop_limit=True,
                   eps=1e-6)
cleaned_df.head()

In [ ]:
# === 1) 读入停复牌，并删除“复牌日期缺失”的事件 ===
sr_path = r"\停复牌日期.csv"
df_sr = pd.read_csv(sr_path)[["股票代码", "停牌日期", "复牌日期"]].copy()

df_sr["code_key"] = df_sr["股票代码"].astype(str).str.extract(r"(\d+)")[0].astype(int)
df_sr["停牌日期"] = pd.to_datetime(df_sr["停牌日期"], errors="coerce")
df_sr["复牌日期"] = pd.to_datetime(df_sr["复牌日期"], errors="coerce")

# 只保留“停牌日期 & 复牌日期都存在”的事件（复牌缺失的认为退市/未复牌，直接丢弃）
df_sr = df_sr.dropna(subset=["停牌日期", "复牌日期"]).copy()

# === 2) 生成“月末处于停牌区间”的(code_key, month_end)标记表 ===
# 月末点满足：停牌日 <= month_end < 复牌日
df_sr["start_m"] = df_sr["停牌日期"] + pd.offsets.MonthEnd(0)
df_sr["end_m"] = df_sr["复牌日期"] + pd.offsets.MonthEnd(-1)
df_sr = df_sr[df_sr["end_m"] >= df_sr["start_m"]].copy()  # 同月内很快复牌的会被自然过滤掉

susp_month = (
    df_sr.assign(month_end=df_sr.apply(lambda r: pd.date_range(r["start_m"], r["end_m"], freq="M"), axis=1))
         .explode("month_end")[["code_key", "month_end"]]
         .drop_duplicates()
)
susp_month["is_suspended"] = True

# === 3) 月度表：把 Code 转成可匹配的 code_key，然后合并停牌标记 ===
cleaned_df["month_end"] = pd.to_datetime(cleaned_df["month_end"])
cleaned_df["code_key"]  = cleaned_df["Code"].astype(str).str.extract(r"(\d+)")[0].astype(int)   

cleaned_df_suspend = cleaned_df.merge(susp_month, on=["code_key", "month_end"], how="left")
cleaned_df_suspend["is_suspended"] = cleaned_df_suspend["is_suspended"].fillna(False)

cleaned_df_suspend.head()


In [ ]:
cleaned_totalshare = cleaned_df_suspend[['month_end','最近交易日','Code','公告日期','统计截止日期','月末收盘价','月末收盘价_后复权','总市值','is_suspended']]
cleaned_totalshare.head()

In [ ]:
def plot_coverage(df_clean, df_base=None,
                  date_col='month_end', code_col='Code',
                  year_step=5, ax=None):
    x = df_clean.loc[~df_clean['is_suspended']].copy()
    x[date_col] = pd.to_datetime(x[date_col])

    clean_cnt = x.groupby(date_col)[code_col].nunique().sort_index()

    if df_base is None:
        base_cnt = pd.Series(clean_cnt.max(), index=clean_cnt.index)
    else:
        b = df_base.copy()
        b[date_col] = pd.to_datetime(b[date_col])
        base_cnt = b.groupby(date_col)[code_col].nunique().reindex(clean_cnt.index)

    cover = (clean_cnt / base_cnt * 100).dropna()

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 3))

    ax.fill_between(cover.index, cover.values, 0, color='0.75', linewidth=0)
    ax.plot(cover.index, cover.values, color='0.25', linewidth=1)
    ax.set_ylim(0, 100)
    ax.set_ylabel('数据覆盖度(%)')
    ax.set_xlabel('日期')
    ax.xaxis.set_major_locator(mdates.YearLocator(base=year_step))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(False)

    return cover, ax


In [ ]:
cover, ax = plot_coverage(df_clean=cleaned_totalshare, df_base=filter_df)

# 第三步 分组检验

In [ ]:
# 描述性统计
def size10_mktcap_row(cleaned_totalshare,
                      date_col="month_end",
                      code_col="Code",
                      cap_col="总市值",
                      n_groups=10):
    df = cleaned_totalshare.copy()

    df[date_col] = pd.to_datetime(df[date_col])
    df[cap_col] = pd.to_numeric(df[cap_col], errors="coerce")
    df = df.dropna(subset=[date_col, code_col, cap_col])

    df["mktcap_yi"] = df[cap_col] / 1e8

    def _assign_decile(x):
        x = x.sort_values("mktcap_yi")
        r = x["mktcap_yi"].rank(method="first")
        x["size_grp"] = pd.qcut(r, q=n_groups, labels=np.arange(1, n_groups + 1))
        return x

    df = (df.groupby(date_col, group_keys=False)
            .apply(_assign_decile)
            .reset_index(drop=True))   # <- 关键：消除 month_end 的索引歧义

    g_month = (df.groupby([date_col, "size_grp"])["mktcap_yi"]
                 .mean()
                 .unstack("size_grp"))

    row = g_month.mean(axis=0)
    row.index = ["Small"] + [str(i) for i in range(2, n_groups)] + ["Big"]

    out = pd.DataFrame([row.values], columns=row.index, index=["总市值（亿元）"])
    return out, df, g_month


In [ ]:
size_row, df_with_grp, month_by_grp = size10_mktcap_row(
    cleaned_totalshare.loc[~cleaned_totalshare["is_suspended"]]
)
display(size_row.round(2))


In [ ]:
def size10_table(df,
                 weighting="equal",
                 date_col="month_end",
                 code_col="Code",
                 price_col="月末收盘价_后复权",
                 cap_col="总市值",
                 suspended_col="is_suspended",
                 end_date="2019-12-31",
                 n_groups=10,
                 sort_lag=1,
                 nw_lags=6,
                 return_ts=False):

    d = df[[date_col, code_col, price_col, cap_col, suspended_col]].copy()
    d[date_col] = pd.to_datetime(d[date_col])
    d[price_col] = pd.to_numeric(d[price_col], errors="coerce")
    d[cap_col]   = pd.to_numeric(d[cap_col], errors="coerce")
    d = d.dropna(subset=[date_col, code_col, price_col, cap_col])

    if end_date is not None:
        d = d[d[date_col] <= pd.to_datetime(end_date)]

    # is_suspended 统一转 bool（兼容 True/False 或 "True"/"False"）
    d[suspended_col] = d[suspended_col].astype(str).str.lower().isin(["true","1","t","yes","y"])

    d = d.sort_values([code_col, date_col])
    d["_mnum"] = d[date_col].dt.year * 12 + d[date_col].dt.month

    # ===== 核心：停牌平滑收益 =====
    def _smooth_ret(x):
        x = x.sort_values(date_col).copy()
        p   = x[price_col].to_numpy(float)
        m   = x["_mnum"].to_numpy(int)
        sus = x[suspended_col].to_numpy(bool)

        ret = np.full(len(x), np.nan)
        trade_idx = np.where((~sus) & ~np.isnan(p))[0]   # 仅用“非停牌月”的价格做锚点

        for a, b in zip(trade_idx[:-1], trade_idx[1:]):
            k = m[b] - m[a]  # 间隔月份数（例如停牌3个月后复牌 -> k=4）
            if k <= 0:
                continue
            rm = (p[b] / p[a]) ** (1.0 / k) - 1.0        # 平滑后的“月收益”
            ret[a+1:b+1] = rm                             # 分摊到区间内每个月（含复牌月）

        x["ret"] = ret
        return x

    d = d.groupby(code_col, group_keys=False).apply(_smooth_ret).reset_index(drop=True)

    # 分组与权重用滞后市值 cap_{t-1}
    d["cap_lag"] = d.groupby(code_col)[cap_col].shift(sort_lag)
    d = d.dropna(subset=["ret", "cap_lag"])

    def _grp_one_month(x):
        r = x["cap_lag"].rank(method="first")
        x["grp"] = pd.qcut(r, q=n_groups, labels=np.arange(1, n_groups + 1))
        return x

    d = d.groupby(date_col, group_keys=False).apply(_grp_one_month)
    d["grp"] = d["grp"].astype(int)

    if weighting.lower() in ["equal", "ew"]:
        ts = d.groupby([date_col, "grp"])["ret"].mean().unstack("grp")
    elif weighting.lower() in ["value", "vw", "cap"]:
        def _vw(x):
            w = x["cap_lag"].to_numpy()
            r = x["ret"].to_numpy()
            sw = w.sum()
            return np.nan if sw == 0 else (w @ r) / sw
        ts = d.groupby([date_col, "grp"], group_keys=False).apply(_vw).unstack("grp")
    else:
        raise ValueError("weighting 只能取 'equal' 或 'value'")

    ts["SB"] = ts[1] - ts[n_groups]

    def _nw_tstat_mean(x):
        x = x.dropna()
        if len(x) < 10:
            return np.nan
        X = np.ones((len(x), 1))
        res = sm.OLS(x.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})
        return float(res.tvalues[0])

    mean_pct = ts.mean() * 100
    tvals = ts.apply(_nw_tstat_mean)

    cols = ["Small"] + [str(i) for i in range(2, n_groups)] + ["Big", "Small-Big"]
    out = pd.DataFrame(index=["Mean(%)", "t(NW)"], columns=cols, dtype=float)

    out.loc["Mean(%)", "Small"] = mean_pct[1]
    out.loc["t(NW)",  "Small"]  = tvals[1]
    for g in range(2, n_groups):
        out.loc["Mean(%)", str(g)] = mean_pct[g]
        out.loc["t(NW)",  str(g)]  = tvals[g]
    out.loc["Mean(%)", "Big"]       = mean_pct[n_groups]
    out.loc["t(NW)",  "Big"]        = tvals[n_groups]
    out.loc["Mean(%)", "Small-Big"] = mean_pct["SB"]
    out.loc["t(NW)",  "Small-Big"]  = tvals["SB"]

    if return_ts:
        return out, ts, d
    return out


In [ ]:
panelA, tsA, dA = size10_table(cleaned_totalshare, weighting="equal",
                               end_date="2025-12-31", return_ts=True)

panelB, tsB, dB = size10_table(cleaned_totalshare, weighting="value",
                               end_date="2025-12-31", return_ts=True)

display(panelA.round(4))
display(panelB.round(4))


In [ ]:
tsA

In [ ]:
tsB

In [ ]:
print(type(tsA), getattr(tsA, "shape", None), getattr(tsA, "index", None)[:3])


In [ ]:
def plot_size_factor(ts, title="规模因子累计收益率", small=1, big=10, ls="SB", start=None, end=None, figsize=(9,6)):
    """
    ts: 月度收益（小数），index=month_end(DatetimeIndex)，列包含 1..10 和 'SB'
    累计收益率：cum = (1+r).cumprod() - 1
    """

    ts = ts.copy()
    ts.index = pd.to_datetime(ts.index)
    ts = ts.sort_index()
    if start: ts = ts.loc[pd.to_datetime(start):]
    if end:   ts = ts.loc[:pd.to_datetime(end)]

    # 兼容列名是 int 或 str
    def pick(c):
        if c in ts.columns: return c
        if isinstance(c, int) and str(c) in ts.columns: return str(c)
        if isinstance(c, str) and c.isdigit() and int(c) in ts.columns: return int(c)
        raise KeyError(f"找不到列：{c}")

    small, big = pick(small), pick(big)

    if ls not in ts.columns:
        ts[ls] = ts[small] - ts[big]

    cum_small = (1 + ts[small].fillna(0)).cumprod() - 1
    cum_big   = (1 + ts[big].fillna(0)).cumprod() - 1
    cum_ls    = (1 + ts[ls].fillna(0)).cumprod() - 1

    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True)

    # Panel A
    axes[0].plot(cum_small.index, cum_small.values, label="市值最小组", linewidth=1.6)
    axes[0].plot(cum_big.index,   cum_big.values,   label="市值最大组", linestyle="--", linewidth=1.6)
    axes[0].set_title("Panel A：市值最小组与市值最大组")
    axes[0].set_ylabel("累计收益率")
    axes[0].legend(frameon=False)
    axes[0].grid(True, alpha=0.25)

    # Panel B
    axes[1].plot(cum_ls.index, cum_ls.values, label="市值多空组合（Small−Big）", linewidth=1.6)
    axes[1].set_title("Panel B：市值多空组合")
    axes[1].set_ylabel("累计收益率")
    axes[1].set_xlabel("日期")
    axes[1].grid(True, alpha=0.25)

    axes[1].xaxis.set_major_locator(mdates.YearLocator(base=2))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.suptitle(title, y=0.98)
    fig.autofmt_xdate()
    plt.tight_layout(rect=[0,0,1,0.96])
    plt.save('')
    return fig, axes

# 用法：



In [ ]:
plot_size_factor(tsA, title="规模因子累计收益率（等权重）",end="2025-12-31")
plot_size_factor(tsB, title="规模因子累计收益率（市值加权）",end="2025-12-31")
